# Reinforcement Learning Comparison: Q-Learning vs Deep Q-Network (DQN)

This notebook provides a detailed comparison between Q-learning (a tabular method) and Deep Q-Network (DQN) in a GridWorld environment with wind and various terrain types.

## Key Findings
- **Q-learning**: 99.8% success rate (500 episodes)
- **DQN**: 3.8% success rate (500 episodes) 

We'll explore why such a significant performance gap exists between these methods and when each would be appropriate to use.

## Import Required Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import re
import glob
import json
from matplotlib.ticker import MaxNLocator
import matplotlib.patches as mpatches

# Set plot style
plt.style.use('ggplot')
sns.set(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 12

## Define Training Results

We'll manually define our training results since we're working with the summary statistics from the terminal output.

In [ ]:
# Define training results for both algorithms
results = {
    'Q-learning': {
        'success_rate': 99.8,
        'avg_reward': 85.54,
        'avg_steps': 13.80,
        'final10_success_rate': 100.0,
        'final10_avg_reward': 88.30,
        'episodes': 500
    },
    'DQN': {
        'success_rate': 3.8,
        'avg_reward': -285.08,
        'avg_steps': 198.91,
        'final10_success_rate': 0.0,
        'final10_avg_reward': -320.20,
        'episodes': 500
    },
    'Initial DQN': {
        'success_rate': 1.0,
        'avg_reward': -350.11,
        'avg_steps': 199.02,
        'final10_success_rate': 0.0,
        'final10_avg_reward': -373.40,
        'episodes': 500
    }
}

# Convert to pandas DataFrame for easier plotting
results_df = pd.DataFrame(results).T
results_df = results_df.reset_index().rename(columns={'index': 'Algorithm'})
results_df

## Visualizing Performance Metrics

Let's create some visualizations to compare the performance of Q-learning vs DQN.

In [ ]:
# Plotting Success Rate Comparison
plt.figure(figsize=(12, 6))
bar_width = 0.35
index = np.arange(3)

# Filter out 'Initial DQN' for main metrics
main_df = results_df[results_df['Algorithm'] != 'Initial DQN']
dqn_improved = results_df[results_df['Algorithm'] == 'DQN'].iloc[0]['success_rate']
dqn_initial = results_df[results_df['Algorithm'] == 'Initial DQN'].iloc[0]['success_rate']

# Create bar chart
bars1 = plt.bar(index[0:2], main_df['success_rate'], bar_width, label='Overall Success Rate', color='#3498db')
bars2 = plt.bar(index[0:2] + bar_width, main_df['final10_success_rate'], bar_width, label='Final 10 Episodes Success Rate', color='#2ecc71')

# Add labels and title
plt.xlabel('Algorithm')
plt.ylabel('Success Rate (%)')
plt.title('Success Rate Comparison: Q-learning vs DQN', fontsize=14)
plt.xticks(index + bar_width/2, main_df['Algorithm'])
plt.ylim(0, 105) # Ensure we can see the full 100%

# Add text annotations
for i, bar in enumerate(bars1):
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2., height + 2,
            f'{height:.1f}%',
            ha='center', va='bottom')
    
for i, bar in enumerate(bars2):
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2., height + 2,
            f'{height:.1f}%',
            ha='center', va='bottom')

plt.legend()
plt.tight_layout()
plt.show()

# Add an inset to show DQN's improvement
plt.figure(figsize=(10, 6))
bars = plt.bar(['Initial DQN', 'Optimized DQN'], [dqn_initial, dqn_improved], color=['#e74c3c', '#3498db'])
plt.ylabel('Success Rate (%)')
plt.title('DQN Improvement After Optimization', fontsize=14)
plt.ylim(0, 5)

# Add text annotations
for bar in bars:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2., height + 0.1,
            f'{height:.1f}%',
            ha='center', va='bottom')
            
plt.tight_layout()
plt.show()

In [ ]:
# Plotting Average Reward Comparison
plt.figure(figsize=(12, 6))

# Create bar chart for average rewards
bars1 = plt.bar(index[0:2], main_df['avg_reward'], bar_width, label='Overall Average Reward', color='#3498db')
bars2 = plt.bar(index[0:2] + bar_width, main_df['final10_avg_reward'], bar_width, label='Final 10 Episodes Average Reward', color='#2ecc71')

# Add labels and title
plt.xlabel('Algorithm')
plt.ylabel('Average Reward')
plt.title('Average Reward Comparison: Q-learning vs DQN', fontsize=14)
plt.xticks(index + bar_width/2, main_df['Algorithm'])

# Add a horizontal line at y=0
plt.axhline(y=0, color='gray', linestyle='-', alpha=0.3)

# Add text annotations
for i, bar in enumerate(bars1):
    height = bar.get_height()
    y_pos = height + 5 if height > 0 else height - 20
    plt.text(bar.get_x() + bar.get_width()/2., y_pos,
            f'{height:.1f}',
            ha='center', va='bottom')
    
for i, bar in enumerate(bars2):
    height = bar.get_height()
    y_pos = height + 5 if height > 0 else height - 20
    plt.text(bar.get_x() + bar.get_width()/2., y_pos,
            f'{height:.1f}',
            ha='center', va='bottom')

plt.legend()
plt.tight_layout()
plt.show()

# Plotting Average Steps Comparison
plt.figure(figsize=(10, 6))
plt.bar(main_df['Algorithm'], main_df['avg_steps'], color='#9b59b6')
plt.xlabel('Algorithm')
plt.ylabel('Average Steps per Episode')
plt.title('Average Steps Comparison: Q-learning vs DQN', fontsize=14)

# Add text annotations
for i, v in enumerate(main_df['avg_steps']):
    plt.text(i, v + 5, f"{v:.1f}", ha='center')
    
plt.tight_layout()
plt.show()

## Combined Performance Analysis

Let's create a more comprehensive view by normalizing and plotting all metrics together to visualize the massive performance gap.

In [ ]:
# Create a radar chart to compare performance across all metrics
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.path import Path
from matplotlib.splines import Spline
from matplotlib.patches import Circle, RegularPolygon
from matplotlib.path import Path
from matplotlib.projections.polar import PolarAxes
from matplotlib.projections import register_projection
from matplotlib.transforms import Affine2D

def radar_factory(num_vars, frame='circle'):
    """Create a radar chart with `num_vars` axes."""
    # Calculate evenly-spaced axis angles
    theta = np.linspace(0, 2*np.pi, num_vars, endpoint=False)

    class RadarAxes(PolarAxes):
        name = 'radar'
        # Use 1 line segment to connect specified points
        RESOLUTION = 1

        def __init__(self, *args, **kwargs):
            super().__init__(*args, **kwargs)
            # Rotate plot such that the first axis is at the top
            self.set_theta_zero_location('N')

        def fill(self, *args, closed=True, **kwargs):
            """Override fill so that line is closed by default"""
            return super().fill(closed=closed, *args, **kwargs)

        def plot(self, *args, **kwargs):
            """Override plot so that line is closed by default"""
            lines = super().plot(*args, **kwargs)
            for line in lines:
                self._close_line(line)
            return lines

        def _close_line(self, line):
            x, y = line.get_data()
            # FIXME: markers at x[0], y[0] get doubled-up
            if x[0] != x[-1]:
                x = np.append(x, x[0])
                y = np.append(y, y[0])
                line.set_data(x, y)

        def set_varlabels(self, labels):
            self.set_thetagrids(np.degrees(theta), labels)

        def _gen_axes_patch(self):
            # The Axes patch must be centered at (0.5, 0.5) and of radius 0.5
            # in axes coordinates.
            if frame == 'circle':
                return Circle((0.5, 0.5), 0.5)
            elif frame == 'polygon':
                return RegularPolygon((0.5, 0.5), num_vars,
                                      radius=.5, edgecolor="k")
            else:
                raise ValueError("Unknown value for 'frame': %s" % frame)

        def _gen_axes_spines(self):
            if frame == 'circle':
                return super()._gen_axes_spines()
            elif frame == 'polygon':
                # spine_type must be 'left', 'right', 'top', 'bottom', or `circle`.
                spine = Spine(axes=self,
                              spine_type='circle',
                              path=Path.unit_regular_polygon(num_vars))
                # unit_regular_polygon gives a polygon of radius 1 centered at
                # (0, 0) but we want a polygon of radius 0.5 centered at (0.5,
                # 0.5) in axes coordinates.
                spine.set_transform(Affine2D().scale(.5).translate(.5, .5)
                                    + self.transAxes)
                return {'polar': spine}
            else:
                raise ValueError("Unknown value for 'frame': %s" % frame)

    register_projection(RadarAxes)
    return theta

# Define metrics we want to compare
metrics = ['Success Rate (%)', 'Avg Reward (normalized)', 'Steps Efficiency (%)']
theta = radar_factory(len(metrics), frame='polygon')

# Prepare data
data = [
    ['Q-learning', [
        results['Q-learning']['success_rate'],  # Success rate
        # Normalize rewards between 0 and 1
        (results['Q-learning']['avg_reward'] - min(results['Q-learning']['avg_reward'], results['DQN']['avg_reward'])) / 
        (max(results['Q-learning']['avg_reward'], 1) - min(results['Q-learning']['avg_reward'], results['DQN']['avg_reward'])),
        # Convert steps to efficiency (fewer steps is better)
        100 * (1 - results['Q-learning']['avg_steps'] / 200)  # 200 is max steps
    ]],
    ['DQN', [
        results['DQN']['success_rate'],  # Success rate
        # Normalize rewards between 0 and 1
        (results['DQN']['avg_reward'] - min(results['Q-learning']['avg_reward'], results['DQN']['avg_reward'])) / 
        (max(results['Q-learning']['avg_reward'], 1) - min(results['Q-learning']['avg_reward'], results['DQN']['avg_reward'])),
        # Convert steps to efficiency (fewer steps is better)
        100 * (1 - results['DQN']['avg_steps'] / 200)  # 200 is max steps
    ]],
]

# Create the figure
fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(projection='radar'))

# Plot each algorithm
colors = ['#2ecc71', '#3498db']
for d, color in zip(data, colors):
    ax.plot(theta, d[1], color=color, linewidth=2, label=d[0])
    ax.fill(theta, d[1], facecolor=color, alpha=0.25)

# Set labels and title
ax.set_varlabels(metrics)
ax.set_title('Algorithm Performance Comparison', fontsize=14, weight='bold', y=1.1)

# Add a legend
ax.legend(loc='upper right', bbox_to_anchor=(0.1, 0.1))

# Add gridlines
ax.grid(True)

# Add percentage markers
ax.set_yticks([0, 20, 40, 60, 80, 100])
ax.set_yticklabels(['0%', '20%', '40%', '60%', '80%', '100%'])
ax.set_ylim(0, 100)

plt.tight_layout()
plt.show()